In [1]:
import pandas as pd
import json
import os
import deepsig
from IPython.display import display

In [2]:
cols = ['dataset', 'method', 'fitness_rule', 'fitness', 'ACC', 'MCC', 'f1_score', 'avg_odds_diff', 'stat_par_diff', 'eq_opp_diff']

In [3]:
mlp_baseline_results = pd.read_csv('simple_mlp_results.csv')
mlp_baseline_results.replace({'simple_mlp_initializer': r'MLP'}, inplace=True)

mlp_standard_l2_results = pd.read_csv('mlp_standard_l2_results.csv')
mlp_standard_l2_results.replace({'mlp_standard_l2_initializer': r'MLP+L2'}, inplace=True)

mlp_featurewise_l2_results = pd.read_csv('mlp_featurewise_l2_results.csv')
mlp_featurewise_l2_results.replace({'mlp_featurewise_l2_initializer': r'MLP+L2^{(1)}'}, inplace=True)

mlp_preg_results = pd.read_csv('mlp_preg_results.csv')
mlp_preg_results.replace({'mlp_preg_initializer': r'MLP+SDR$_{\rho}$'}, inplace=True)

mlp_sreg_results = pd.read_csv('mlp_sreg_results.csv')
mlp_sreg_results.replace({'mlp_sreg_initializer': r'MLP+SDR$_{\rho_s}$'}, inplace=True)

mlp_kreg_results = pd.read_csv('mlp_kreg_results.csv')
mlp_kreg_results.replace({'mlp_kreg_initializer': r'MLP+SDR$_{\tau}$'}, inplace=True)

mlp_xi_reg_results = pd.read_csv('mlp_xi_reg_results.csv')
mlp_xi_reg_results.replace({'mlp_xi_reg_initializer': r'MLP+SDR$_{\xi}$'}, inplace=True)

ftl_baseline_results = pd.read_csv('ftl_mlp_results.csv')
ftl_baseline_results.replace({'ftl_mlp_initializer': r'FTL'}, inplace=True)

ftl_xi_reg_results = pd.read_csv('ftl_mlp_xi_reg_results.csv')
ftl_xi_reg_results.replace({'ftl_mlp_xi_reg_initializer': r'FTL+SDR$_{\xi}$'}, inplace=True)


results = pd.concat([mlp_baseline_results, mlp_standard_l2_results, mlp_featurewise_l2_results, mlp_preg_results, mlp_sreg_results, mlp_kreg_results, mlp_xi_reg_results, ftl_baseline_results, ftl_xi_reg_results], ignore_index=True)

/var/folders/z5/rq0dv5jj45qccc_tc39171cm0000gn/T/ipykernel_50770/3877943148.py:29: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([mlp_baseline_results, mlp_standard_l2_results, mlp_featurewise_l2_results, mlp_preg_results, mlp_sreg_results, mlp_kreg_results, mlp_xi_reg_results, ftl_baseline_results, ftl_xi_reg_results], ignore_index=True)


In [4]:
results.replace({'adult_dataset_reader': 'Adult Income', 'compas_dataset_reader': 'Compas Recidivism', 'german_dataset_reader': 'German Credit', 'bank_dataset_reader': 'Bank Marketing'}, inplace=True)
results.rename(columns={'avg_odds_diff': 'Equalized Odds', 'stat_par_diff': 'Statistical Parity', 'eq_opp_diff': 'Equal Opportunity', 'MCC': 'Mathew Correlation', 'ACC': 'Accuracy'}, inplace=True)

In [5]:
fitness_rules_target_metrics = {
    'mcc_parity': {'performance': 'Mathew Correlation', 'fairness': 'Statistical Parity'},
    'mcc_opportunity': {'performance': 'Mathew Correlation', 'fairness': 'Equal Opportunity'},
    'mcc_odds': {'performance': 'Mathew Correlation', 'fairness': 'Equalized Odds'},
    'acc_parity': {'performance': 'Accuracy', 'fairness': 'Statistical Parity'},
    'acc_opportunity': {'performance': 'Accuracy', 'fairness': 'Equal Opportunity'},
    'acc_odds': {'performance': 'Accuracy', 'fairness': 'Equalized Odds'}
}

fitness_rules_target_metrics = {
    'mcc_parity': ('Mathew Correlation', 'Statistical Parity'),
    'mcc_opportunity': ('Mathew Correlation', 'Equal Opportunity'),
    'mcc_odds': ('Mathew Correlation', 'Equalized Odds'),
    'acc_parity': ('Accuracy', 'Statistical Parity'),
    'acc_opportunity': ('Accuracy', 'Equal Opportunity'),
    'acc_odds': ('Accuracy', 'Equalized Odds')
}
fitness_rules_abvr = {
    'mcc_parity': 'Max(MCC - Stat. Parity)',
    'mcc_opportunity': 'Max(MCC - Eq. Odds)',
    'mcc_odds': 'Max(MCC - Eq. Opp.)',
    'acc_parity': 'Max(Acc - Stat. Parity)',
    'acc_opportunity': 'Max(Acc - Eq. Odds)',
    'acc_odds': 'Max(Acc - Eq. Opp.)'
}

results['Performance'] = 0
results['Fairness'] = 0
results['Fitness Rule'] = ''
for fitness_rule, (performance_metric, fairness_metric) in fitness_rules_target_metrics.items():
    results.loc[results.fitness_rule == fitness_rule,'Performance'] = results.loc[results.fitness_rule == fitness_rule,performance_metric]
    results.loc[results.fitness_rule == fitness_rule,'Fairness'] = results.loc[results.fitness_rule == fitness_rule,fairness_metric]
    results.loc[results.fitness_rule == fitness_rule,'Fitness Rule Abvr'] = fitness_rules_abvr[fitness_rule]
    results.loc[results.fitness_rule == fitness_rule,'Fitness Rule'] = 'Max(%s - %s)' % fitness_rules_target_metrics[fitness_rule]

/var/folders/z5/rq0dv5jj45qccc_tc39171cm0000gn/T/ipykernel_50770/604014855.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.58070551 0.52018535 0.29655197 0.47568263 0.57556857 0.4830075
 0.28451811 0.45966628 0.56390249 0.50399739 0.29592974 0.40844112
 0.57035482 0.51343466 0.26678803 0.42633265 0.56603901 0.56247684
 0.25744332 0.37124592 0.58820616 0.5265673  0.30763987 0.33949448
 0.59003607 0.52782167 0.27597802 0.17785946 0.57784061 0.51747652
 0.27693768 0.4019954  0.58246514 0.50669613 0.28033877 0.37713259
 0.58299125 0.52769824 0.29070644 0.44543433 0.57307304 0.5301395
 0.32123244 0.20458494 0.5709584  0.56062881 0.29330813 0.41761288
 0.57130473 0.48822413 0.58829309 0.52447717 0.28703819 0.27574114
 0.55289194 0.52620217 0.33787721 0.45311014 0.58453141 0.49519859
 0.25498836 0.43004191 0.58562544 0.5062429  0.28661199 0.23156974
 0.58324647 0.51350231 0.29560743 0.38019877 0.5836501  0.5246249

In [6]:
datasets = ['Adult Income', 'Bank Marketing', 'Compas Recidivism','German Credit']
datasets

['Adult Income', 'Bank Marketing', 'Compas Recidivism', 'German Credit']

In [7]:
fitness_rules = ['mcc_parity', 'mcc_opportunity', 'mcc_odds', 'acc_parity', 'acc_opportunity', 'acc_odds']
fitness_rules

['mcc_parity',
 'mcc_opportunity',
 'mcc_odds',
 'acc_parity',
 'acc_opportunity',
 'acc_odds']

In [8]:
methods = [r'MLP', r'MLP+L2', r'MLP+L2^{(1)}', r'MLP+SDR$_{\rho}$', r'MLP+SDR$_{\rho_s}$', r'MLP+SDR$_{\tau}$', r'MLP+SDR$_{\xi}$', r'FTL', r'FTL+SDR$_{\xi}$']

In [9]:
grouped_results = results\
    .groupby(['fitness_rule', 'dataset', 'method'])\
    .agg({'fitness': ['mean', 'std', 'count'], 'Performance': ['mean', 'std'], 'Fairness': ['mean', 'std']})\
    #.sort_values(by=['fitness_rule', 'dataset', ('fitness','mean')], ascending=[False, True, False])
grouped_results['formatted_fitness'] = grouped_results.apply(lambda row: f"${row[('fitness', 'mean')]:.3f} (\pm{row[('fitness', 'std')]:.2f})$", axis=1)
grouped_results['formatted_performance'] = grouped_results.apply(lambda row: f"${row[('Performance', 'mean')]:.3f} (\pm{row[('Performance', 'std')]:.2f})$", axis=1)
grouped_results['formatted_fairness'] = grouped_results.apply(lambda row: f"${row[('Fairness', 'mean')]:.3f} (\pm{row[('Fairness', 'std')]:.2f})$", axis=1)
grouped_results

fitness                  \
                                                   mean       std count   
fitness_rule dataset       method                                         
acc_odds     Adult Income  FTL                 0.797546  0.022527    25   
                           FTL+SDR$_{\xi}$     0.805417  0.018968    15   
                           MLP+L2              0.762226  0.018550    15   
                           MLP+L2^{(1)}        0.758089  0.019553    15   
                           MLP+SDR$_{\rho_s}$  0.772452  0.020815    16   
...                                                 ...       ...   ...   
mcc_parity   German Credit MLP+L2^{(1)}        0.274434  0.090499    15   
                           MLP+SDR$_{\rho_s}$  0.192318  0.107623    16   
                           MLP+SDR$_{\rho}$    0.254748  0.071677    16   
                           MLP+SDR$_{\tau}$    0.255848  0.079738    14   
                           MLP+SDR$_{\xi}$     0.245571  0.070406    15   

                                              Performance            Fairness  \
                                                     mean       std      mean   
fitness_rule dataset       method                                               
acc_odds     Adult Income  FTL                   0.840876  0.007559  0.043330   
                           FTL+SDR$_{\xi}$       0.843169  0.005611  0.037752   
                           MLP+L2                0.848631  0.002860  0.086405   
                           MLP+L2^{(1)}          0.846884  0.003580  0.088795   
                           MLP+SDR$_{\rho_s}$    0.848646  0.002916  0.076194   
...                                                   ...       ...       ...   
mcc_parity   German Credit MLP+L2^{(1)}          0.360494  0.073887  0.086060   
                           MLP+SDR$_{\rho_s}$    0.291751  0.073053  0.099432   
                           MLP+SDR$_{\rho}$      0.341448  0.067708  0.086701   
                           MLP+SDR$_{\tau}$      0.341519  0.054298  0.085671   
                           MLP+SDR$_{\xi}$       0.329368  0.050637  0.083796   

                                                         formatted_fitness  \
                                                    std                      
fitness_rule dataset       method                                            
acc_odds     Adult Income  FTL                 0.023855  $0.798 (\pm0.02)$   
                           FTL+SDR$_{\xi}$     0.017930  $0.805 (\pm0.02)$   
                           MLP+L2              0.018986  $0.762 (\pm0.02)$   
                           MLP+L2^{(1)}        0.019361  $0.758 (\pm0.02)$   
                           MLP+SDR$_{\rho_s}$  0.020507  $0.772 (\pm0.02)$   
...                                                 ...                ...   
mcc_parity   German Credit MLP+L2^{(1)}        0.055244  $0.274 (\pm0.09)$   
                           MLP+SDR$_{\rho_s}$  0.060881  $0.192 (\pm0.11)$   
                           MLP+SDR$_{\rho}$    0.042336  $0.255 (\pm0.07)$   
                           MLP+SDR$_{\tau}$    0.056780  $0.256 (\pm0.08)$   
                           MLP+SDR$_{\xi}$     0.060963  $0.246 (\pm0.07)$   

                                              formatted_performance  \
                                                                      
fitness_rule dataset       method                                     
acc_odds     Adult Income  FTL                    $0.841 (\pm0.01)$   
                           FTL+SDR$_{\xi}$        $0.843 (\pm0.01)$   
                           MLP+L2                 $0.849 (\pm0.00)$   
                           MLP+L2^{(1)}           $0.847 (\pm0.00)$   
                           MLP+SDR$_{\rho_s}$     $0.849 (\pm0.00)$   
...                                                             ...   
mcc_parity   German Credit MLP+L2^{(1)}           $0.360 (\pm0.07)$   
                           MLP+SDR$_{\rho_s}$     $0.292 (\

In [10]:
selected_columns = ['formatted_fitness', 'formatted_performance', 'formatted_fairness']
for fitness_rule in fitness_rules:
    grouped_results.loc[fitness_rule][selected_columns].to_latex(f'tables/grouped_results_{fitness_rule}_crp.tex')
     #.to_latex(f'tables/grouped_results_{fitness_rule}_crp.tex', columns=selected_columns))